[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/06_your_own_classifier.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Your own classifier's scores

Allow about 20 minutes. What does today's analysis say about the
classifier you trained this week? You will join its scores to the CinC
labels, freeze a policy on the development recordings, and only then look at
the evaluation recordings. Labels are real CinC labels. The scores are yours,
or a simulated demonstration column if you have none.

The action is **send this recording for AF review**, priced as in the core
notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0, in illustrative loss units.
Replace each `...` and run the checker below it; if you are stuck for about
10 minutes, use the stuck switch below the function, where there is one, or open the worked copy in `extension/solutions/`.
A later cell that needs an unfinished problem prints `Waiting for ...`: once that
problem is done or switched, click in the waiting cell, choose **Runtime > Run
before**, then run the waiting cell.

**Which scores to bring.** Export a CSV with a header line `recording,score`
and one row per CinC recording, with the score your model gives for AF. The
notebook assigns your recordings to the course's development and evaluation
splits.

The reveal at the end is a held-out evaluation only if no evaluation
recording influenced your model in any way: not in training, preprocessing or
feature selection, hyperparameter tuning, early stopping, checkpoint or model
selection, recalibration, or an earlier choice of threshold. A validation set
you used for early stopping, for example, does not qualify. If you cannot say
that, leave `EVALUATION_UNTOUCHED = False` below and read the result as a
retrospective policy analysis: it shows how a frozen policy behaves on those
recordings, not how it would perform on new ones.

In [ ]:
import numpy as np
from IPython.display import display
from clinical_utility import data, reference, own_scores, plots
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)

# Upload your CSV in Colab's Files pane and give its name here, or paste its lines below.
SCORES_FILE = ""
PASTED_SCORES = """
"""
# True only if no evaluation recording influenced your model, as described above.
EVALUATION_UNTOUCHED = False

scores, note = own_scores.read_scores(SCORES_FILE, PASTED_SCORES)
print(note)
print(len(scores), "rows read. First rows:", scores[:3])

## Join the scores to the labels

`labels` maps every CinC recording ID to its reference label: `A` (AF), `N`,
`O` or `~` (noise). The join must not guess. Check each row in this order:

| Check | What to do with the row |
|---|---|
| Its ID appears more than once in your file | Exclude every copy; list the ID once in `duplicates`. |
| Its ID is not a CinC recording | Exclude it; list it in `unknown`. |
| Its score is not a number between zero and one | Exclude it; list it in `out_of_range`. |
| Its label is noise, `~` | Exclude it; add one to `noise`. |
| Otherwise | Keep `{"recording", "score", "af_present"}`, with `af_present` true only for `A`. |

### Problem A: Make

Write `join_scores(scores, labels)`, returning a dictionary with the keys
`records`, `duplicates`, `unknown`, `out_of_range` and `noise`, as in the
starter. Keep rows in their input order. A NaN score fails both `score >= 0`
and `score <= 1`.

In [ ]:
def join_scores(scores, labels):
    return {"records": ..., "duplicates": ..., "unknown": ..., "out_of_range": ..., "noise": ...}
_ = check.run("own_join", join_scores)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# join_scores = check.use_reference("join_scores")

In [ ]:
# The join, then what each split can support. Evaluation outcomes stay hidden until the reveal.
own = {"development": [], "evaluation": []}
if check.ready(join_scores=join_scores):
    joined = join_scores(scores, data.reference_labels())
    for key in ("duplicates", "unknown", "out_of_range"):
        print(f"{key}: {len(joined[key])}", joined[key][:5])
    print("noise recordings excluded:", joined["noise"])
    own = own_scores.with_splits(joined["records"])
    own_scores.describe_splits(own)

## The core analysis on development only

### Problem B: Predict

On your development recordings, will the cutoff that maximises J be lower or
higher than the minimum-loss cutoff, or the same? Store `"lower"`, `"higher"`
or `"same"`. The checker works it out from your own scores, and accepts any
answer that a pair of tied optima supports. If J is undefined on your
recordings, it says so.

In [ ]:
j_prediction = ...
_ = check.run("own_j", j_prediction, own["development"])

In [ ]:
# The minimum-loss and maximum-J policies on development.
best = None
if not own["development"]:
    check.waiting("development recordings: finish Problem A, or bring scores for "
                  "recordings in the development split")
else:
    sweep = reference.sweep_thresholds(own["development"], costs)
    best = reference.best_thresholds(sweep)[0]
    marks = {"Minimum loss": best}
    best_j = own_scores.max_j(sweep)  # None, with the reason printed, when J is undefined
    if best_j:
        marks["Maximum J"] = best_j
    for label, row in marks.items():
        print(f"{label:12}  cutoff {row['threshold']:.3f}  reviews {row['reviews']}  "
              f"missed AF {row['missed_af']}  loss {row['total_loss']:.0f}")
    display(plots.objective_figure(sweep, marks))

### Acting on a stated 0.2

If your scores were calibrated probabilities, reviewing at 0.2 would be the
right policy at these prices. The next cell compares the decisions and the
loss at a stated 0.2 with the empirical optimum. Every cutoff in an interval
makes the same decisions, so a minimum-loss cutoff far from 0.2 can still
make the same decisions as 0.2 when no score lies between them. The bootstrap
interval holds both cutoffs fixed, so it leaves out the optimism of choosing
the optimum on these same recordings.

In [ ]:
if not best:
    check.waiting("the development analysis above, which needs development recordings")
else:
    compared = own_scores.stated_versus_optimum(own["development"], costs, .2, best)
    for label, row in (("At a stated 0.2", compared["stated"]), ("Minimum loss", compared["best"])):
        print(f"{label:16} reviews {row['reviews']}  missed AF {row['missed_af']}  "
              f"loss {row['total_loss']:.0f}")
    print(f"Any cutoff {own_scores.interval_text(compared['interval'])} makes the minimum-loss "
          f"decisions; 0.2 is {'inside' if compared['same_decisions'] else 'outside'} that interval.")
    print(f"Extra loss at 0.2: {compared['extra_loss']:.0f} (95% bootstrap interval "
          f"{compared['extra_loss_low']:.0f} to {compared['extra_loss_high']:.0f}).")
    at_0_2 = reference.confusion_counts(reference.evaluate_policy(own["development"], .2, costs))
    review_all = reference.confusion_counts(reference.evaluate_policy(own["development"], 0, costs))
    print("Net benefit at a stated 0.2:", round(reference.net_benefit(at_0_2, .2), 4),
          " review everyone:", round(reference.net_benefit(review_all, .2), 4))

In [ ]:
# Can the scores be read as probabilities? The reliability summary, then the logistic fit.
if not best:
    check.waiting("the development analysis above, which needs development recordings")
else:
    summary = reference.calibration_summary(own["development"])
    display(view.reliability({"Your scores": {
        "predicted": np.array([b["predicted"] for b in summary["points"]]),
        "observed": np.array([b["observed"] for b in summary["points"]]),
        "n": np.array([b["n"] for b in summary["points"]])}}))
    print("Brier:", round(summary["brier"], 4))
    fit = own_scores.calibration_fit(own["development"])  # None for anything undefined, and why
    for key, value in fit.items():
        if value is not None:
            print(f"Calibration {key}: {value:.3f}")

Read the outputs as in the core. Extra loss at 0.2 that is large and whose
interval excludes zero is a reason to examine calibration. It does not prove
miscalibration: sampling variation also moves an empirical optimum. Judge
calibration from the reliability summary, and from the intercept and slope
when they can be estimated. Does J choose a policy you could defend at these
prices?

## Freeze, then look

### Problem C: Predict

Set `final_threshold` from your development analysis, for example
`best["threshold"]`, and predict the total loss on your evaluation recordings.
Only their number is known so far. Scale by the number of recordings: the two
splits differ in size. Only then run the reveal cell.

In [ ]:
final_threshold = ...
predicted_evaluation_loss = ...
frozen = check.run("own_freeze", (final_threshold, predicted_evaluation_loss),
                   own["development"], own["evaluation"])

In [ ]:
# Reveal: your frozen policy on development and on evaluation.
if not frozen:
    check.waiting("Problem C: freeze a policy on development")
elif not own["evaluation"]:
    print("There are no evaluation recordings among your scores, so there is nothing to reveal.")
else:
    report = {split: reference.policy_summary(own[split], final_threshold, costs)
              for split in ("development", "evaluation")}
    plots.show_report(report, predicted_evaluation_loss)
    print(own_scores.evaluation_label(EVALUATION_UNTOUCHED))

## Before you finish

- Say whether any evaluation recording influenced your model, and how you know.
- Report the cutoff, the reviews and the missed AF beside the AUC.
- Say what acting on a stated 0.2 costs, and what the reliability summary shows.
- Compare the evaluation loss with your prediction and explain the gap.

**Discuss:** What would you report beside AUC for this classifier, and what
would a service need to know before acting on it?